In [1]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [2]:
BASE_DIR = Path.cwd().parent   # se estiveres em notebooks/
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

In [3]:
df = pd.read_csv(DATA_DIR / "train.csv")
df.head()
dt= pd.read_csv(DATA_DIR / "test.csv")


In [4]:
features = [
    'ChiefComplaint',
    'age',
    'PulseRate',
    'RespiratoryRate',
    'gender']

target = 'TriageGrade'

categorical_features = ['ChiefComplaint', 'gender']
numerical_features = ['age', 'PulseRate', 'RespiratoryRate']

df = df[features + [target]]
dt = dt[features + [target]]

X_train = df[features]
y_train = df[target]

X_test = dt[features]
y_test = dt[target]


In [5]:
df

,ChiefComplaint,age,PulseRate,RespiratoryRate,gender,TriageGrade
0,S43.0,24,90.0,16.0,Male,3
1,I10,88,83.0,17.0,Female,3
2,S09.90XA,63,NaN,NaN,Male,2
3,R20.2,39,NaN,NaN,Male,2
4,R31.9,64,NaN,NaN,Male,2
...,...,...,...,...,...,...
93151,T79.9,35,89.0,18.0,Male,3
93152,M79.60,20,NaN,NaN,Female,2
93153,K92.2,79,NaN,NaN,Female,2
93154,U07.1,31,NaN,NaN,Male,2


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numerical_features
        )
    ]
)


In [7]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

param_dist = {
    "classifier__n_estimators": range(10, 51, 5), # 10 a 50, de 5 em 5
    "classifier__max_depth": range(1, 21),
    "classifier__criterion": ["gini", "entropy"]
}

search1 = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)

search1.fit(X_train, y_train)
print("Best parameters:", search1.best_params_)


Best parameters: {'classifier__n_estimators': 45, 'classifier__max_depth': 14, 'classifier__criterion': 'gini'}


In [8]:
rf_pipeline_best = search1.best_estimator_

y_pred = rf_pipeline_best.predict(X_test)

print("RANDOM FOREST")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


RANDOM FOREST
Accuracy: 0.74581356405247
              precision    recall  f1-score   support

           1       0.26      0.64      0.37      2058
           2       0.88      0.67      0.76     16191
           3       0.91      0.92      0.92      6864
           4       0.68      0.82      0.75      3548
           5       0.00      0.00      0.00         3

    accuracy                           0.75     28664
   macro avg       0.55      0.61      0.56     28664
weighted avg       0.82      0.75      0.77     28664



Trying to make gender into a numerical feature

In [9]:
import os

os.makedirs("models", exist_ok=True)
joblib.dump({
    "model": rf_pipeline_best,
    "features": features
},"models/random_forest.joblib")



['models/random_forest.joblib']